# Test-Set Evaluation


## 1. Setup

In [3]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error,mean_squared_error,r2_score
warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

## 2. Load all artifacts

In [6]:
required_files={
    "xgb":"models/xgb.pkl",
    "lgbm":"models/lgbm.pkl",
    "rf":"models/rf.pkl",
    "ridge":"models/ridge.pkl",
    "scaler":"models/meta_scaler.pkl",
    "feat_cols":"models/feature_columns.json",
    "baseline":"results/baseline_metrics.json"
}
xgb_model=joblib.load("models/xgb.pkl")
lgbm_model=joblib.load("models/lgbm.pkl")
rf_model=joblib.load("models/rf.pkl")
ridge=joblib.load("models/ridge.pkl")
scaler=joblib.load("models/meta_scaler.pkl")
with open("models/feature_columns.json") as f:
    feature_columns=json.load(f)
with open("results/baseline_metrics.json") as f:
    baseline_results=json.load(f)

## 3. Load test set

In [7]:
test = pd.read_csv("../Data/Clean Data/test.csv")
print(f"Test shape: {test.shape}")
y_test=test["trans_value"]
X_test=test.drop(columns=["trans_value","transaction_num"],errors="ignore")
X_test.columns=[c.replace(" ", "_").replace("-", "_") for c in X_test.columns]
missing_cols=[c for c in feature_columns if c not in X_test.columns]
extra_cols=[c for c in X_test.columns if c not in feature_columns]
if missing_cols:
    raise ValueError(f"Test set is missing columns: {missing_cols}")
X_test=X_test[feature_columns]
print(f"Aligned to {X_test.shape[1]} feature columns matching training")

Test shape: (79129, 32)
Aligned to 30 feature columns matching training


## 4. Stacked predictions with confidence intervals

In [8]:
xgb_preds=xgb_model.predict(X_test)
lgbm_preds=lgbm_model.predict(X_test)
rf_preds=rf_model.predict(X_test)
# Stack and pass through the meta learner
meta_test_raw=np.column_stack([xgb_preds,lgbm_preds,rf_preds])
meta_test_scaled=scaler.transform(meta_test_raw)
final_preds=ridge.predict(meta_test_scaled)
# Uncertainty proxy from base learner disagreement
base_std=meta_test_raw.std(axis=1)
ci_low=final_preds-1.96*base_std
ci_high=final_preds+1.96*base_std

## 5. Overall test metrics

In [9]:
overall_mape=mean_absolute_percentage_error(y_test,final_preds)
overall_rmse=float(np.sqrt(mean_squared_error(y_test,final_preds)))
overall_r2=r2_score(y_test,final_preds)
overall_results=pd.DataFrame({
    "Metric":["MAPE","RMSE","R2"],
    "Value":[overall_mape,overall_rmse,overall_r2]
})
overall_results.to_csv("results/overall_metrics.csv",index=False)
print("Final test results:")
print(f"MAPE: {overall_mape*100}%")
print(f"RMSE: {overall_rmse} AED")
print(f"R^2: {overall_r2}")

Final test results:
MAPE: 13.213370090043252%
RMSE: 789503.6161535917 AED
R^2: 0.9274765633536569


## 6. Segmentation by property type

In [10]:
prop_segments={
    "Land":test["prop_type_Land"]==1,
    "Unit":test["prop_type_Unit"]==1,
    "Villa":test["prop_type_Villa"]==1
}
prop_rows=[]
for name,mask in prop_segments.items():
    if mask.sum()==0:
        continue
    seg_y,seg_pred=y_test[mask],final_preds[mask]
    prop_rows.append({
        "Property_Type":name,
        "Count":int(mask.sum()),
        "MAPE":mean_absolute_percentage_error(seg_y,seg_pred),
        "RMSE":float(np.sqrt(mean_squared_error(seg_y,seg_pred))),
        "R2":r2_score(seg_y,seg_pred)
    })
prop_df=pd.DataFrame(prop_rows)
prop_df.to_csv("results/property_type_metrics.csv",index=False)
print("Property Type Metrics:")
print(prop_df.to_string(index=False))

Property Type Metrics:
Property_Type  Count     MAPE         RMSE       R2
         Land   4212 0.249620 1.773728e+06 0.905751
         Unit  66365 0.124344 6.079123e+05 0.916226
        Villa   8465 0.131588 1.083974e+06 0.850925


## 7. Segmentation by year

In [11]:
year_rows=[]
for year in sorted(test["date_year"].unique()):
    mask=test["date_year"]==year
    if mask.sum()<50:
        continue
    seg_y,seg_pred=y_test[mask],final_preds[mask]
    year_rows.append({
        "Year":int(year),
        "Count":int(mask.sum()),
        "MAPE":mean_absolute_percentage_error(seg_y,seg_pred),
        "RMSE":float(np.sqrt(mean_squared_error(seg_y,seg_pred))),
        "R2":r2_score(seg_y,seg_pred)
    })
year_df=pd.DataFrame(year_rows)
year_df.to_csv("results/year_metrics.csv",index=False)
print("Year Metrics")
print(year_df.to_string(index=False))

Year Metrics
 Year  Count     MAPE          RMSE       R2
 2025  12757 0.105648 598000.180217 0.944497
 2026  66372 0.137224 821210.670051 0.924949


## 8. Baseline vs. stacked ensemble

In [12]:
comparison=pd.DataFrame([
    {
        "Model":"Baseline XGBoost (CV)",
        "MAPE":baseline_results["MAPE"],
        "RMSE":baseline_results["RMSE"],
        "R2":baseline_results["R2"]
    },
    {
        "Model":"Stacked Ensemble (Test)",
        "MAPE":overall_mape,
        "RMSE":overall_rmse,
        "R2":overall_r2
    }
])
comparison["MAPE_pct"]=(comparison["MAPE"]*100).round(2)
comparison.to_csv("results/baseline_vs_stacked.csv",index=False)
improvement_pp=(baseline_results["MAPE"]-overall_mape)*100
print("Baseline vs Stacked:")
print(comparison.to_string(index=False))
print(f"\nMAPE improvement: {improvement_pp} percentage points")

Baseline vs Stacked:
                  Model     MAPE          RMSE       R2  MAPE_pct
  Baseline XGBoost (CV) 0.189342 929229.214583 0.871604     18.93
Stacked Ensemble (Test) 0.132134 789503.616154 0.927477     13.21

MAPE improvement: 5.720818536827693 percentage points


## 9. Save per-row predictions

In [13]:
predictions_df=pd.DataFrame({
    "actual":y_test.values,
    "predicted":final_preds,
    "ci_low":ci_low,
    "ci_high":ci_high,
    "base_spread":base_std,
    "xgb_pred":xgb_preds,
    "lgbm_pred":lgbm_preds,
    "rf_pred":rf_preds
})
predictions_df.to_csv("results/test_predictions.csv",index=False)

## 10. Summary

In [14]:
print(f"""
Headline Metrics:
Test MAPE:{overall_mape*100}%
Test RMSE: {overall_rmse} AED
Test R^2: {overall_r2}

Baseline Comparison:
Baseline XGBoost MAPE: {baseline_results["MAPE"]*100}%
Stacked Ensemble MAPE: {overall_mape*100}%
Improvement: {improvement_pp} percentage points

By Property Type - MAPE
{prop_df[["Property_Type","Count","MAPE"]].to_string(index=False)}

By Year - MAPE
{year_df[["Year", "Count", "MAPE"]].to_string(index=False)}
""")



Headline Metrics:
Test MAPE:13.213370090043252%
Test RMSE: 789503.6161535917 AED
Test R^2: 0.9274765633536569

Baseline Comparison:
Baseline XGBoost MAPE: 18.934188626870945%
Stacked Ensemble MAPE: 13.213370090043252%
Improvement: 5.720818536827693 percentage points

By Property Type - MAPE
Property_Type  Count     MAPE
         Land   4212 0.249620
         Unit  66365 0.124344
        Villa   8465 0.131588

By Year - MAPE
 Year  Count     MAPE
 2025  12757 0.105648
 2026  66372 0.137224

